In [0]:
%run ./00_config_and_logging

In [0]:
dbutils.widgets.text("load_type", "full", "Load type (full / incremental)")
dbutils.widgets.text("repo_path", "/Workspace/Users/naumaniqbal987@gmail.com/cyber-threat-intel", "Repo root path")

load_type = dbutils.widgets.get("load_type")
repo_path = dbutils.widgets.get("repo_path")

folder = "full_load" if load_type == "full" else "incremental"
data_base = f"{repo_path}/data/sample_raw/{folder}"
print("Reading from:", data_base)

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime

run_id = new_run_id()
start = datetime.now()

epss_file = "epss_full_sample.csv" if load_type == "full" else "epss_incremental_sample.csv"
epss_path = f"{data_base}/{epss_file}"

try:
    epss_df = spark.read \
        .option("header", "true") \
        .option("comment", "#") \
        .schema(epss_schema) \
        .csv(epss_path)

    epss_df = epss_df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(epss_file)) \
        .withColumn("load_type", lit(load_type))

    epss_df.write.format("delta").mode("append").saveAsTable(BRONZE_EPSS)

    write_log(run_id, "Raw-to-Bronze", load_type, epss_file, BRONZE_EPSS,
              start, "Success", rows_inserted=epss_df.count())
    print("EPSS rows written:", epss_df.count())
except Exception as e:
    write_log(run_id, "Raw-to-Bronze", load_type, epss_file, BRONZE_EPSS,
              start, "Failure", error_message=str(e))
    raise

In [0]:
from pyspark.sql.functions import explode, col

run_id = new_run_id()
start = datetime.now()

kev_file = "kev_full_sample.json" if load_type == "full" else "kev_incremental_sample.json"
kev_path = f"{data_base}/{kev_file}"

try:
    kev_raw_df = spark.read \
        .option("multiline", "true") \
        .schema(kev_root_schema) \
        .json(kev_path)

    kev_df = kev_raw_df.select(explode(col("vulnerabilities")).alias("vuln")) \
        .select("vuln.*")

    kev_df = kev_df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(kev_file)) \
        .withColumn("load_type", lit(load_type))

    kev_df.write.format("delta").mode("append").saveAsTable(BRONZE_KEV)

    write_log(run_id, "Raw-to-Bronze", load_type, kev_file, BRONZE_KEV,
              start, "Success", rows_inserted=kev_df.count())
    print("KEV rows written:", kev_df.count())
except Exception as e:
    write_log(run_id, "Raw-to-Bronze", load_type, kev_file, BRONZE_KEV,
              start, "Failure", error_message=str(e))
    raise

In [0]:
run_id = new_run_id()
start = datetime.now()

nvd_file = "nvd_cves_full_sample.json" if load_type == "full" else "nvd_cves_incremental_sample.json"
nvd_path = f"{data_base}/{nvd_file}"

try:
    nvd_raw_df = spark.read \
        .option("multiline", "true") \
        .schema(nvd_root_schema) \
        .json(nvd_path)

    nvd_df = nvd_raw_df.select(explode(col("vulnerabilities")).alias("item")) \
        .select("item.cve.*")

    nvd_df = nvd_df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(nvd_file)) \
        .withColumn("load_type", lit(load_type))

    nvd_df.write.format("delta").mode("append").saveAsTable(BRONZE_NVD)

    write_log(run_id, "Raw-to-Bronze", load_type, nvd_file, BRONZE_NVD,
              start, "Success", rows_inserted=nvd_df.count())
    print("NVD rows written:", nvd_df.count())
except Exception as e:
    write_log(run_id, "Raw-to-Bronze", load_type, nvd_file, BRONZE_NVD,
              start, "Failure", error_message=str(e))
    raise

In [0]:
run_id = new_run_id()
start = datetime.now()

asset_file = "cortexa_assets_full_sample.csv" if load_type == "full" else "cortexa_assets_incremental_sample.csv"
asset_path = f"{data_base}/{asset_file}"
schema_to_use = asset_schema if load_type == "full" else asset_incremental_schema

try:
    asset_df = spark.read \
        .option("header", "true") \
        .schema(schema_to_use) \
        .csv(asset_path)

    asset_df = asset_df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(asset_file)) \
        .withColumn("load_type", lit(load_type))

    asset_df.write.format("delta").mode("append").saveAsTable(BRONZE_ASSETS)

    write_log(run_id, "Raw-to-Bronze", load_type, asset_file, BRONZE_ASSETS,
              start, "Success", rows_inserted=asset_df.count())
    print("Asset rows written:", asset_df.count())
except Exception as e:
    write_log(run_id, "Raw-to-Bronze", load_type, asset_file, BRONZE_ASSETS,
              start, "Failure", error_message=str(e))
    raise

In [0]:
display(spark.table(LOG_TABLE))

In [0]:
# check count where cvid has substring less than 2021
display(spark.sql(f"select count(*) from {BRONZE_EPSS} where cve like 'CVE-2021%'"))
# spark.table(BRONZE_EPSS).schema